In [ ]:
import os
import gc
import certifi
import pandas as pd
import numpy as np
import scanpy as sc
import diopy
import bbknn
import scvi
import scrublet as scr
import cellrank as cr
import matplotlib.pyplot as plt
import scanpy.external as sce
import scvelo as scv
import scib
from cellrank.kernels import PseudotimeKernel
from cellrank.kernels import CytoTRACEKernel
from scib_metrics.benchmark import Benchmarker

sc.logging.print_header()


In [ ]:
# data dir
input_dir = " "
output_dir = " "
os.listdir(input_dir)

In [ ]:
## pre work
tumor_code = "BRCA"
cluster_method = "leiden"
bbknn_ridge = True
batch_method = "bbknn"
regress = False
resolution = 5
random_state = 123


In [ ]:
## data read
# scRNA_current = diopy.input.read_h5(file = f"{input_dir}/{tumor_code}/scRNA_bbknn.h5")
# scRNA_origin = sc.read_h5ad(f"{input_dir}/{tumor_code}/scRNA_in_batch.h5ad")
scRNA_current = sc.read_h5ad(f"{input_dir}/{tumor_code}/scRNA_remove_batch.h5ad")

## dir create
output_file = f"{output_dir}/{tumor_code}"
os.makedirs(output_file, exist_ok=True)

In [ ]:
print(f"############################# {tumor_code} process ##########################################")

## cluster
if batch_method == "harmony":
    sc.pp.neighbors(scRNA_current, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=40)
elif batch_method == "scvi":
    sc.pp.neighbors(scRNA_current, use_rep="X_scvi")
elif batch_method == "scanorama":
    sc.pp.neighbors(scRNA_current, use_rep="X_scanorama")
elif batch_method == "none":
    sc.pp.neighbors(scRNA_current, use_rep="X_pca", n_neighbors=15, n_pcs=40)

if cluster_method == "leiden":
    sc.tl.leiden(scRNA_current, resolution=resolution, n_iterations=-1, random_state=random_state)
else:
    sc.tl.louvain(scRNA_current, resolution=resolution, random_state=random_state)
    
sc.tl.umap(scRNA_current, random_state=random_state)

## bbknn ridge
if bbknn_ridge == True:
    bbknn.ridge_regression(scRNA_current, batch_key=["sample_ID","study_ID","tissue_type"], confounder_key = "leiden")
    sc.tl.pca(scRNA_current, svd_solver='arpack', use_highly_variable=True, n_comps=50, random_state=random_state)
    sc.external.pp.bbknn(scRNA_current, batch_key= "sample_ID", n_pcs = 50)
    sc.tl.leiden(scRNA_current, resolution=resolution, n_iterations=-1, random_state=random_state)
    sc.tl.umap(scRNA_current, random_state=random_state)

In [ ]:
sc.pl.umap(scRNA_current, color='leiden',show = False, legend_loc='on data')
plt.savefig(f'{output_file}/scRNA_umap_cluster.png', dpi=3000)

plt.figure(figsize=(10, 10))
sc.pl.umap(scRNA_current, color='sample_ID',show = False, legend_loc='none', frameon=False, title='')
plt.savefig(f'{output_file}/scRNA_umap_sample.png', dpi=3000)

sc.pl.umap(scRNA_current, color='tumor_status',show = False)
plt.savefig(f'{output_file}/scRNA_umap_tumor_status.png', dpi=3000)

sc.pl.umap(scRNA_current, color='study_ID',show = False)
plt.savefig(f'{output_file}/scRNA_umap_study.png', dpi=3000)

sc.pl.umap(scRNA_current, color='tissue_type',show = False)

# markers = ["EPCAM","KRT18","COL1A1","DCN","CD3D","NKG7","CD79A","CD79B","C1QC","FCN1","CD1C",
#            "PECAM1","VWF","TPSAB1","KIT","CSF3R","MYH11","CNN1","RGS5","CD36","ALB","APOC3"]
markers = ["EPCAM","KRT18","COL1A1","DCN","CD3D","NKG7","CD79A","CD79B","C1QC","FCN1","CD1C",
           "PECAM1","VWF","TPSAB1","KIT","CSF3R","MYH11","CNN1","RGS5","CD36","ALPL","RUNX2","MLANA","SPP1","CD8A","CD4","FOXP3"]
sc.pl.umap(scRNA_current, color=markers)

In [ ]:
# data save
# scRNA_current.X = scRNA_current.layers["counts"].copy()
diopy.output.write_h5(scRNA_current, file = f"{output_file}/scRNA_cluster.h5",save_X=False)
scRNA_current.write_h5ad(f"{output_file}/scRNA_cluster.h5ad", compression="gzip")